# hscredit 审计复现

基线为 0.1.3，提交 bab104b19216309f10ed09184888aa69dac30ff5。此 Notebook 复现分析报告中的语义差异并验证真实数据评分卡流程，不修改库代码，不连接数据库。JSON 和工作簿写入仓库 `.codex-temp/audit-20261104`，不导出客户明细。

部分探针预期记录异常或指标差异，这些正是审计发现，不能把所有单元格成功运行理解为问题已修复。执行真实样例需要本地 `examples/hscredit_yyp.xlsx`。

In [1]:
import json
import importlib
import sys
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'hscredit' / '__init__.py').is_file())
sys.path.insert(0, str(ROOT))
import hscredit as h

OUT = ROOT / '.codex-temp' / 'audit-20261104'
OUT.mkdir(exist_ok=True)
evidence = {'python': sys.version, 'versions': {p: version(p) for p in ['numpy', 'pandas', 'scikit-learn', 'scipy', 'joblib']}}
evidence['imported_versions'] = {p: importlib.import_module(p).__version__ for p in ['numpy', 'pandas', 'sklearn', 'scipy', 'joblib']}
X = pd.DataFrame({'x': np.linspace(-3, 3, 80)})
y = np.array([0, 1, 0, 0, 0, 1, 0, 1] * 10)
m = LogisticRegression().fit(X, y)

def probe(name, fn):
    try:
        evidence[name] = fn()
    except Exception as exc:
        evidence[name] = {'exception': type(exc).__name__, 'message': str(exc)}
    print(name, json.dumps(evidence[name], ensure_ascii=False), flush=True)



## 定义可独立重跑的探针

`group_cv` 直接检查调参器 fit 使用的内部划分校验路径，避免运行无关 trial；其余探针使用公共入口。

In [2]:
def calibration():
    models = [h.ProbabilityCalibrator(model=m, calib_ratio=None).fit(X, y, **kw) for kw in
              [{}, {'sample_weight': np.where(y == 1, 30., 1.)}, {'sample_weight': [-1]}, {'misspelled_parameter': 123}]]
    p = models[0].predict_proba(X)
    return {'max_probability_differences': [float(np.max(np.abs(c.predict_proba(X) - p))) for c in models],
            'invalid_weight_accepted': True, 'unknown_parameter_accepted': True}

def report_json():
    r = h.ModelReport(m, X_train=X, y_train=y, n_jobs=1, explain_config={'enabled': False})
    return json.dumps(r.to_dict(), ensure_ascii=False)

def weighted_report():
    A = pd.DataFrame({'x': np.linspace(-4, 1, 200)})
    B = pd.DataFrame({'x': np.linspace(2, 7, 200)})
    labels = (np.arange(200) % 3 == 0).astype(int)
    model = LogisticRegression().fit(A, labels)
    r = h.ModelReport(model, datasets={'train': (A, labels), 'test': (B, labels)}, n_jobs=1,
        sample_weight={k: np.where(labels == 1, 20., 1.) for k in ['train', 'test']}, explain_config={'enabled': False})
    metric = r.get_metrics().set_index('统计项')
    bins = {k: r.get_bin_table(k, max_n_bins=3) for k in ['train', 'test']}
    return {'summary_bad_rate': float(metric.at['坏样本率', 'train']),
            'bin_total_bad_rate': float(bins['train'].iloc[-1]['坏样本率']),
            'bin_labels': {k: v['分箱标签'].tolist() for k, v in bins.items()}}

def encoder():
    A = pd.DataFrame({'id': [f'u{i}' for i in range(100)]}); labels = np.tile([0, 1], 50)
    return {mode: roc_auc_score(labels, h.TargetEncoder(features=['id'], training_mode=mode,
        random_state=42, n_jobs=1).fit_transform(A, labels)['id']) for mode in ['in_sample', 'oof']}

def group_cv():
    from hscredit.core.models.tuning import ModelTuner
    A = pd.DataFrame({'x': np.arange(100, dtype=float)}); labels = np.tile([0, 1], 50)
    groups = np.repeat(np.arange(20), 5)
    t = ModelTuner(LogisticRegression(), search_space={'C': [1.]}, cv=3, n_jobs=1, random_state=42)
    # Inspect the exact split validation path used by fit without paying for trial training.
    return {'groups_shared_between_train_validation': [len(set(groups[a]) & set(groups[b]))
            for a, b in t._validated_splits(A, labels, groups)]}

def target_modes():
    frame = X.iloc[:5].copy(); frame['target'] = [0, 1, np.nan, 0, 1]
    result = {}
    for mode in ['explicit_y', 'target_column', 'target_column_del_grey']:
        kw = {'X_train': frame, 'n_jobs': 1, 'explain_config': {'enabled': False}}
        kw.update({'y_train': frame['target']} if mode == 'explicit_y' else {'target': 'target', 'del_grey': mode.endswith('del_grey')})
        r = h.ModelReport(m, **kw)
        result[mode] = r.get_metrics().set_index('统计项')['训练集'].to_dict()
    return result

def eda_status():
    frame = pd.DataFrame({'x': [1, 2, 3, 4]})
    simple = h.eda_summary(frame, target='missing', n_jobs=1)
    structured = h.eda_summary(frame, target='missing', n_jobs=1, return_result=True)
    return {'default_type': type(simple).__name__, 'default_keys': list(simple),
            'structured_complete': structured.complete, 'statuses': structured.status_table()[['章节', '状态']].to_dict('records')}

def report_options():
    report = h.ModelReport(m, X_train=X, y_train=y, metrics=['brier'], n_jobs=1, explain_config={'enabled': False})
    table = report.get_bin_table(amount='nonexistent_amount')
    return {'metrics': report.get_metrics()['统计项'].tolist(),
            'summary_metrics': report.summary().columns.get_level_values(0).unique().tolist(),
            'missing_amount_accepted': True, 'bin_columns': table.columns.tolist()}

def invalid_probability():
    class InvalidProbability:
        classes_ = np.array([0, 1])
        feature_names_in_ = np.array(['x'])
        def predict_proba(self, frame):
            p = np.linspace(-1, 2, len(frame))
            return np.column_stack([1 - p, p])
    report = h.ModelReport(InvalidProbability(), X_train=X, y_train=y, n_jobs=1, explain_config={'enabled': False})
    return {'out_of_range_probability_accepted': True, 'metrics': report.get_metrics().to_dict('records')}

def real_workflow():
    frame = pd.read_excel(ROOT / 'examples' / 'hscredit_yyp.xlsx')
    columns = ['衡枢鉴真分老客版', '近六个月非银多头机构数', '手机号近一个月非银多头机构数']
    features = frame[columns]; labels = frame['FPD']
    tr, te = train_test_split(np.arange(len(frame)), test_size=.3, random_state=42, stratify=labels)
    binner = h.OptimalBinning(method='quantile', max_n_bins=5, n_jobs=1).fit(features.iloc[tr], labels.iloc[tr])
    encoded = binner.transform(features.iloc[tr], metric='woe')
    card = h.ScoreCard(binner=binner).fit(encoded, labels.iloc[tr])
    predictions = card.predict_proba(features.iloc[te])
    artifact = OUT / 'scorecard.pkl'
    card.save_artifact(artifact)
    restored = h.ScoreCard.load_artifact(artifact)
    difference = float(np.max(np.abs(predictions - restored.predict_proba(features.iloc[te]))))
    report = h.ModelReport(card, datasets={'训练集': (features.iloc[tr], labels.iloc[tr]),
        '测试集': (features.iloc[te], labels.iloc[te])}, n_jobs=1, explain_config={'enabled': False})
    result = {'rows': len(frame), 'columns': len(frame.columns), 'train_rows': len(tr), 'test_rows': len(te),
        'artifact_max_probability_difference': difference, 'metrics': report.get_metrics().to_dict('records')}
    try:
        exported = report.to_excel(str(OUT / 'real-model-report.xlsx'), with_plots=False,
            include_sample_records=False, mode='strict', return_result=True)
        from openpyxl import load_workbook
        workbook = load_workbook(OUT / 'real-model-report.xlsx', read_only=True)
        result.update(report_complete=exported.complete, report_sheets=workbook.sheetnames)
        workbook.close()
    except Exception as exc:
        result['export_error'] = {'type': type(exc).__name__, 'message': str(exc)}
    return result



## 执行并保存结果

固定随机种子。真实数据使用分层随机留出，只验证功能与保存加载一致性，不作为跨期模型效果结论。

In [3]:
for name, fn in [('calibration_parameters', calibration), ('report_json', report_json),
                 ('report_weights_and_bins', weighted_report), ('target_encoding', encoder),
                 ('group_cv', group_cv), ('label_input_modes', target_modes), ('eda_status', eda_status),
                 ('report_options', report_options), ('invalid_probability', invalid_probability),
                 ('real_workflow', real_workflow)]:
    probe(name, fn)
(OUT / ('evidence-py' + str(sys.version_info.major) + str(sys.version_info.minor) + '.json')).write_text(
    json.dumps(evidence, ensure_ascii=False, indent=2, default=str), encoding='utf-8')


calibration_parameters {"max_probability_differences": [0.0, 0.0, 0.0, 0.0], "invalid_weight_accepted": true, "unknown_parameter_accepted": true}


report_json {"exception": "TypeError", "message": "keys must be str, int, float, bool or None, not tuple"}


report_weights_and_bins {"summary_bad_rate": 0.9097080787508486, "bin_total_bad_rate": 0.335, "bin_labels": {"train": ["[-inf, 0.3333)", "[0.3333, 0.3366)", "[0.3366, +inf)", "合计"], "test": ["[-inf, 0.3216)", "[0.3216, 0.3248)", "[0.3248, +inf)", "合计"]}}


target_encoding {"in_sample": 1.0, "oof": 0.444}


group_cv {"groups_shared_between_train_validation": [19, 19, 19]}


label_input_modes {"explicit_y": {"KS": 0.5, "AUC": 0.75, "样本数": 4.0, "坏样本率": 0.5}, "target_column": {"KS": 0.5, "AUC": 0.6666666666666667, "样本数": 5.0, "坏样本率": 0.4}, "target_column_del_grey": {"KS": 0.5, "AUC": 0.75, "样本数": 4.0, "坏样本率": 0.5}}


... (6 outputs left out)

invalid_probability {"out_of_range_probability_accepted": true, "metrics": [{"统计项": "KS", "训练集": 0.046666666666666745}, {"统计项": "AUC", "训练集": 0.5166666666666667}, {"统计项": "样本数", "训练集": 80.0}, {"统计项": "坏样本率", "训练集": 0.375}]}


C:\Users\18306\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


real_workflow {"rows": 970, "columns": 18, "train_rows": 679, "test_rows": 291, "artifact_max_probability_difference": 0.0, "metrics": [{"统计项": "KS", "训练集": 0.19488103821196823, "测试集": 0.24341463414634149}, {"统计项": "AUC", "训练集": 0.6361391492429704, "测试集": 0.6212682926829268}, {"统计项": "样本数", "训练集": 679, "测试集": 291.0}, {"统计项": "坏样本率", "训练集": 0.13991163475699558, "测试集": 0.140893470790378}, {"统计项": "PSI", "训练集": "\\", "测试集": 0.020506325060705082}], "report_complete": true, "report_sheets": ["目录", "1-基本信息", "2-模型性能", "3-入模变量分析", "4-稳定性分析", "5-模型参数", "6-模型部署需求", "7-训练与调参过程"]}


3901